In [4]:
import os
import re
import pandas as pd

In [9]:
def parse_hisat2_summary(file_path):
    """Parses a single HISAT2 summary file and returns a dictionary of stats."""
    stats = {"File Name": os.path.basename(file_path)}

    with open(file_path, "r") as f:
        content = f.read()

    # Regex patterns capture both counts and percentages
    patterns = {
        "Total Reads": r"(\d+) reads; of these:",
        # Concordant 0x
        "Concordant 0x Count": r"(\d+) \([\d\.]+\%\) aligned concordantly 0 times",
        "Concordant 0x (%)": r"\d+ \(([\d\.]+)\%\) aligned concordantly 0 times",
        # Concordant 1x
        "Concordant 1x Count": r"(\d+) \([\d\.]+\%\) aligned concordantly exactly 1 time",
        "Concordant 1x (%)": r"\d+ \(([\d\.]+)\%\) aligned concordantly exactly 1 time",
        # Concordant >1x
        "Concordant >1x Count": r"(\d+) \([\d\.]+\%\) aligned concordantly >1 times",
        "Concordant >1x (%)": r"\d+ \(([\d\.]+)\%\) aligned concordantly >1 times",
        # Discordant 1x
        "Discordant 1x Count": r"(\d+) \([\d\.]+\%\) aligned discordantly 1 time",
        "Discordant 1x (%)": r"\d+ \(([\d\.]+)\%\) aligned discordantly 1 time",
        # Bottom section mates alignment
        "Mates Unaligned": r"(\d+) \([\d\.]+\%\) aligned 0 times\s+\d+ \([\d\.]+\%\) aligned exactly 1 time",
        "Mates 1x": r"(\d+) \([\d\.]+\%\) aligned exactly 1 time",
        "Mates >1x": r"(\d+) \([\d\.]+\%\) aligned >1 times",
        # Overall alignment
        "Overall Alignment Rate (%)": r"([\d\.]+)\% overall alignment rate",
    }

    for key, pattern in patterns.items():
        match = re.search(pattern, content)
        if match:
            if "%" in key or "Rate" in key:
                stats[key] = float(match.group(1))
            else:
                stats[key] = int(match.group(1))
        else:
            stats[key] = None

    return stats

In [11]:
def process_directory(directory_path):
    """Processes all .txt summary files in a directory and returns a DataFrame."""
    all_data = []

    for filename in os.listdir(directory_path):
        if filename.endswith(".txt"):  # Adjust extension if needed
            file_path = os.path.join(directory_path, filename)
            try:
                file_stats = parse_hisat2_summary(file_path)
                # Only add if we successfully parsed the total reads (basic validation)
                if file_stats["Total Reads"] is not None:
                    all_data.append(file_stats)
            except Exception as e:
                print(f"Error processing {filename}: {e}")

    df = pd.DataFrame(all_data)
    return df

In [16]:
cv2017_alignment = process_directory("/work/clh162/henry/results/cv2017/aligned")
yale2025_alignment = process_directory("/work/clh162/henry/results/yale25/aligned")
ru2025_alignment = process_directory("/work/clh162/OysterRNA24/hisat2_align/alignedreads")


In [20]:
## compare mean and std of all alignment %s
pct_cols = [
    "Concordant 0x (%)",
    "Concordant 1x (%)",
    "Concordant >1x (%)",
    "Discordant 1x (%)",
    "Overall Alignment Rate (%)",
]

cv2017_alignment["Assembly"] = "cv2017"
yale2025_alignment["Assembly"] = "yale2025"
ru2025_alignment["Assembly"] = "ru2025"

combined_df = pd.concat(
    [cv2017_alignment, yale2025_alignment, ru2025_alignment], ignore_index=True
)

summary_stats = combined_df.groupby("Assembly")[pct_cols].agg(["mean", "std"])
summary_stats = summary_stats.round(2)
summary_stats

Concordant 0x (%)       Concordant 1x (%)       Concordant >1x (%)  \
                      mean   std              mean   std               mean   
Assembly                                                                      
cv2017               30.35  2.30             63.03  2.06               6.62   
ru2025               30.91  2.27             61.74  2.06               7.35   
yale2025             35.02  2.24             62.04  2.13               2.94   

               Discordant 1x (%)       Overall Alignment Rate (%)        
           std              mean   std                       mean   std  
Assembly                                                                 
cv2017    0.28              3.02  0.27                      77.58  2.53  
ru2025    0.61              1.17  0.11                      76.54  2.50  
yale2025  0.21              0.72  0.07                      72.19  2.47

In [22]:
combined_df.to_csv("/work/clh162/henry/results/alignment_stats.csv")